# U8 · Redes neuronales, imagen y señal — CNN y ViT

> **De la tabla al píxel.** Hasta ahora todo nuestro aprendizaje automático vivía en **tablas**: filas y columnas de `pacientes.csv`. Pero la medicina no cabe entera en una tabla: una radiografía de tórax, una imagen dermatoscópica, un electrocardiograma son **píxeles y señales**, datos "crudos" con estructura espacial o temporal que un modelo tabular no sabe leer. Aquí entramos en la familia que ha protagonizado la última década de la IA: las **redes neuronales**.

> ⚠️ **Aviso de datos.** La parte (A) usa `pacientes.csv`, **sintético** (semilla fija, no representa pacientes reales; la primera celda lo genera sola). Las partes (B) y (C) usan **MedMNIST**, un conjunto de imágenes médicas **público y real** (no inventado), publicado por la comunidad científica para docencia e investigación — lo decimos explícitamente porque es una excepción consciente al hilo sintético del curso.

**El recorrido, de menor a mayor ambición:**
1. **(A) MLP tabular** — una red sencilla sobre `pacientes.csv`, para ver cómo se entrena una red... y comprobar que en tabular **no** le gana a la humilde regresión logística.
2. **(B) CNN sobre imagen médica** — un clasificador de radiografías de tórax con MedMNIST (PneumoniaMNIST), la arquitectura que domina la imagen.
3. **(C) Un modelo ya entrenado** — usar, sin entrenar nada, una red que otros ya afinaron para detectar neumonía. El puente hacia U9.

{% hint style="info" %}
**🖥️ GPU gratis en Colab.** Las partes (B) y (C) rinden mucho mejor con GPU. En Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU* (T4, gratuita). Con MedMNIST, en minutos tienes un clasificador de imagen médica funcionando. Si no tienes GPU a mano, no pasa nada: el notebook está escrito para que **nunca se rompa** — si falta red o GPU, cada celda lo avisa y sigue.
{% endhint %}

[Abrir en Colab](https://colab.research.google.com/drive/1NGzKU1gh2CaN5Cd9sddN7mWqmpang_A_)


## 0. Preparamos los datos tabulares (esta celda se ejecuta sola)

La parte (A) de este cuaderno usa `pacientes.csv`. Como en todo el curso, la primera celda de código **genera los ficheros sintéticos** en la carpeta de trabajo si no existen ya — no hay que descargar nada. Ejecútala una vez (▶ o *Mayús+Enter*) y sigue hacia abajo. Recuerda: son datos **inventados**, sirven para aprender, no para decidir sobre personas.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## Parte (A) · Un MLP sobre datos tabulares

Empezamos por el terreno que ya conocemos —la tabla de pacientes— para entender **cómo se entrena una red** antes de saltar a la imagen. Vamos a usar un **MLP** (perceptrón multicapa): la red más básica, varias capas de neuronas totalmente conectadas, tal como se describe en la unidad. La pregunta que queremos responder es muy concreta: **¿mejora una red a la regresión logística cuando el dato es una tabla?** Ya vimos en U4 que la logística alcanzaba un AUC ≈ 0,84 sobre `evento_cv`. Ese número es nuestra vara de medir.


### A.1 Conoce tus datos: recordamos la tabla

Antes de entrenar nada, recordamos la forma de `pacientes.csv` y cuántos pacientes tuvieron el evento cardiovascular (`evento_cv`). Es la misma tabla de siempre; aquí solo la re-inspeccionamos para tener las variables frescas.


In [ ]:
import pandas as pd

pacientes = pd.read_csv("pacientes.csv")
print("Filas y columnas:", pacientes.shape)
print("\nPrevalencia de evento_cv (proporción de pacientes con evento):")
print(pacientes["evento_cv"].value_counts(normalize=True).round(3))
pacientes.head()

**Lo que vemos:** unas dos décimas partes de los pacientes tuvieron el evento (`evento_cv = 1`); el resto no. Es la misma cohorte sintética que usamos en unidades anteriores. Con esta tabla vamos a entrenar tanto la red como, para comparar, la logística de referencia.


### A.2 Preparamos variables, partición y escalado

Igual que en U4-U5, separamos variables numéricas y categóricas, y partimos en **entrenamiento/test** de forma **estratificada** (para que ambos conjuntos conserven la misma proporción de eventos). Una diferencia importante frente a los modelos de árboles: **las redes neuronales SÍ necesitan las variables numéricas escaladas** (media 0, desviación 1). Sin escalar, una variable en unidades grandes (la glucemia, en cientos) dominaría el entrenamiento sobre otra en unidades pequeñas (el IMC, en decenas), y la red aprendería mal y más despacio.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

num = ["edad", "imc", "ta_sistolica", "ta_diastolica", "glucemia",
       "colesterol_total", "hdl", "antecedentes_familiares", "diabetes"]
cat = ["sexo", "tabaquismo", "actividad_fisica"]

X = pacientes[num + cat]
y = pacientes["evento_cv"]

pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Pacientes en entrenamiento:", Xtr.shape[0])
print("Pacientes en test         :", Xte.shape[0])

**Lo que hemos hecho:** el `ColumnTransformer` estandariza las variables numéricas y convierte las categóricas (sexo, tabaquismo, actividad física) en columnas 0/1 (*one-hot*). Reservamos un 25 % de los pacientes como test, **nunca vistos** durante el entrenamiento, para evaluar con honestidad al final.


### A.3 Entrenamos el MLP

Ahora sí: creamos el MLP con `sklearn.neural_network.MLPClassifier`. Esto **sí corre en un portátil normal**, sin GPU: la tabla es pequeña (unas 20 000 filas, una docena de variables) y la red, modesta —dos capas ocultas de 32 y 16 neuronas—. Encadenamos el preprocesado y la red en un único `Pipeline`, exactamente como hicimos con la logística y el Random Forest en unidades anteriores.


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

mlp = Pipeline([
    ("pre", pre),
    ("clf", MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500,
                           random_state=0, early_stopping=True)),
])
mlp.fit(Xtr, ytr)

prob_mlp = mlp.predict_proba(Xte)[:, 1]
auc_mlp = roc_auc_score(yte, prob_mlp)
print("AUC del MLP en test:", round(auc_mlp, 3))

**Lo que hemos hecho:** `early_stopping=True` es la técnica que menciona la unidad — la red separa una parte del entrenamiento como validación interna y **para sola** cuando deja de mejorar, para no sobreajustar. El número que importa es el **AUC en test**: la capacidad del modelo para ordenar correctamente a los pacientes por riesgo, medida sobre datos que la red no vio al entrenar.


### A.4 El momento de la verdad: MLP contra regresión logística

Entrenamos ahora, sobre la misma partición, la regresión logística de referencia (la de U4) y la comparamos con el MLP en la misma tabla. Es una comparación justa: mismos datos de entrenamiento, mismo test, misma métrica.


In [ ]:
from sklearn.linear_model import LogisticRegression

logistica = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=2000)),
]).fit(Xtr, ytr)

prob_log = logistica.predict_proba(Xte)[:, 1]
auc_log = roc_auc_score(yte, prob_log)

print("AUC regresión logística :", round(auc_log, 3))
print("AUC MLP (red neuronal)  :", round(auc_mlp, 3))

**Lo que significa este resultado — y es el mensaje central de esta parte (A):** el MLP **no mejora** a la regresión logística; a lo sumo iguala su rendimiento, y con más coste de cómputo, más hiperparámetros que ajustar y una interpretabilidad mucho peor (no hay "odds ratio" que leer en una red). Es exactamente lo que anuncia la unidad: **las redes brillan con datos no tabulares** (imagen, señal, texto); en una **tabla** de pacientes, donde el riesgo es aproximadamente log-aditivo, **gana lo simple**. Entrenar un MLP aquí ha sido un buen ejercicio para entender cómo aprende una red — pero no era la herramienta que este problema necesitaba.


{% hint style="success" %}
**💡 Idea clave de la parte (A)**

No hay una "mejor familia de modelos" en abstracto. Hay una familia **adecuada al tipo de dato**. Para tablas clínicas seguimos recomendando lo que ya sabes de U4-U5. El terreno donde las redes de verdad marcan la diferencia es el que exploramos a continuación: la **imagen médica**.
{% endhint %}


## Parte (B) · Una CNN sobre imagen médica real (MedMNIST)

Cambiamos de terreno por completo. En lugar de una tabla de números, trabajamos con **imágenes**: radiografías de tórax en miniatura del conjunto **PneumoniaMNIST**, parte de la colección **MedMNIST v2**. Es imagen médica **real y pública** (no sintética), pensada exactamente para esto: aprender y prototipar en minutos, incluso sin una GPU potente.

**Qué vamos a hacer:** instalar `medmnist`, cargar PneumoniaMNIST (clasificación binaria: ¿neumonía o normal?), mirar algunas imágenes con su etiqueta, y entrenar una **CNN pequeña** que aprenda a distinguirlas, viendo su matriz de confusión.

{% hint style="warning" %}
**⚠️ Esta celda necesita descargar datos (y, para ir rápido, GPU)**

Todo el bloque (B) está pensado para ejecutarse en **Google Colab con GPU**, donde hay red para descargar el paquete y las imágenes, y donde una CNN entrena en segundos. Por eso cada celda de esta parte está envuelta en un `try/except`: si en el entorno donde lees esto **no hay red o el proceso es demasiado lento**, verás un aviso explicando qué se ha omitido y **por qué** — el notebook sigue funcionando sin errores. En Colab, con GPU activada, todo se ejecuta de verdad.
{% endhint %}


### B.1 Instalamos MedMNIST

`medmnist` es la librería que da acceso a toda la colección con una sola línea. La instalamos con `pip`. Si no hay conexión a internet en este entorno, lo detectamos y avisamos, sin detener el resto del cuaderno.


In [ ]:
# Instalación de MedMNIST. Pensada para Colab (con red). Si falla o tarda demasiado, lo explicamos y seguimos.
try:
    try:
        import medmnist  # si ya está instalado (p. ej. Colab en una ejecución previa), no hace falta red
    except ImportError:
        import subprocess, sys as _sys
        resultado = subprocess.run(
            [_sys.executable, "-m", "pip", "install", "-q", "medmnist"],
            capture_output=True, text=True, timeout=20,
        )
        if resultado.returncode != 0:
            raise RuntimeError(resultado.stderr[-300:] if resultado.stderr else "pip devolvió un error")
        import medmnist
    MEDMNIST_OK = True
    print("medmnist instalado correctamente. Versión:", medmnist.__version__)
except Exception as e:
    MEDMNIST_OK = False
    print("Esta celda está pensada para Google Colab con GPU; aquí se omite:",
          "no se ha podido instalar/importar medmnist (sin red, timeout, o el paquete no está disponible). Motivo:", repr(e))
    print("En Colab, ejecuta: !pip install medmnist  — y todo funcionará con red y GPU disponibles.")

**Qué acabamos de comprobar:** si estás en Colab con conexión, `MEDMNIST_OK` será `True` y ya tienes la librería lista. Si estás en un entorno sin red (o la instalación tarda demasiado), el mensaje te lo dice con claridad y la variable queda en `False` — las siguientes celdas leerán esa variable para decidir si intentan trabajar de verdad o si también se omiten con el mismo aviso.


### B.2 Cargamos PneumoniaMNIST y miramos algunas imágenes

PneumoniaMNIST contiene radiografías de tórax pediátricas en miniatura (28×28 píxeles), ya etiquetadas como *neumonía* o *normal*. La primera vez que se pide, `medmnist` **descarga** el conjunto (unos pocos MB) desde su repositorio oficial. Cargamos el subconjunto de entrenamiento y test, y mostramos unas cuantas imágenes con su etiqueta para "verlas" antes de modelar — la misma regla de siempre en este curso: comprensión del dato primero.


In [ ]:
try:
    if not MEDMNIST_OK:
        raise RuntimeError("medmnist no disponible (ver celda anterior)")

    import medmnist
    from medmnist import PneumoniaMNIST
    import numpy as np
    import matplotlib.pyplot as plt

    info = medmnist.INFO["pneumoniamnist"]
    print("Tarea:", info["description"])
    print("Clases:", info["label"])

    train_ds = PneumoniaMNIST(split="train", download=True)
    test_ds = PneumoniaMNIST(split="test", download=True)
    print("\nImágenes de entrenamiento:", len(train_ds))
    print("Imágenes de test          :", len(test_ds))

    fig, ejes = plt.subplots(1, 6, figsize=(12, 2.5))
    for i, ax in enumerate(ejes):
        img, etiqueta = train_ds[i]
        ax.imshow(img, cmap="gray")
        nombre = info["label"][str(int(etiqueta[0]))]
        ax.set_title(nombre, fontsize=9)
        ax.axis("off")
    plt.suptitle("PneumoniaMNIST: radiografías de tórax de ejemplo")
    plt.tight_layout(); plt.show()

except Exception as e:
    print("Esta celda está pensada para Google Colab con GPU; aquí se omite:",
          "no se han podido descargar/cargar las imágenes de PneumoniaMNIST (sin red o medmnist no disponible). Motivo:", repr(e))
    print("En Colab, con red disponible, esta celda descarga el dataset automáticamente y muestra las radiografías.")

**Lo que veríamos (en Colab):** una fila de pequeñas radiografías de tórax en escala de grises, cada una rotulada como *pneumonia* o *normal*. A simple vista, para un no especialista, algunas diferencias son sutiles — precisamente el tipo de patrón visual que una CNN puede aprender a distinguir con más consistencia que un vistazo rápido.


### B.3 Construimos y entrenamos una CNN pequeña

Con las imágenes cargadas, construimos una **CNN sencilla** con Keras/TensorFlow: un par de capas convolucionales (que aprenden filtros locales — bordes, texturas) seguidas de una capa de reducción y una capa final que clasifica. Es la arquitectura mínima descrita en la unidad: convolución → reducción → clasificación. Entrenamos pocas épocas — con GPU, en menos de un minuto tienes un clasificador funcionando.


In [ ]:
try:
    if not MEDMNIST_OK:
        raise RuntimeError("medmnist no disponible (ver celdas anteriores)")

    import tensorflow as tf
    from tensorflow.keras import layers, models
    import numpy as np

    # Preparamos los arrays de imágenes y etiquetas, normalizados a [0, 1]
    Xtr_img = np.stack([np.array(train_ds[i][0]) for i in range(len(train_ds))]).astype("float32") / 255.0
    ytr_img = np.array([int(train_ds[i][1][0]) for i in range(len(train_ds))])
    Xte_img = np.stack([np.array(test_ds[i][0]) for i in range(len(test_ds))]).astype("float32") / 255.0
    yte_img = np.array([int(test_ds[i][1][0]) for i in range(len(test_ds))])
    Xtr_img = Xtr_img[..., np.newaxis]
    Xte_img = Xte_img[..., np.newaxis]

    cnn = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(16, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    cnn.summary()

    historial = cnn.fit(Xtr_img, ytr_img, epochs=5, batch_size=64,
                         validation_split=0.1, verbose=2)

except Exception as e:
    print("Esta celda está pensada para Google Colab con GPU; aquí se omite:",
          "no se ha podido entrenar la CNN (falta TensorFlow, faltan las imágenes cargadas, o no hay cómputo/tiempo suficiente). Motivo:", repr(e))
    print("En Colab, con GPU activada (Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU), este entrenamiento tarda menos de un minuto.")

**Lo que hemos hecho (en Colab):** apilamos dos bloques de convolución + reducción — la jerarquía de la unidad: primero bordes y contrastes simples, luego patrones más complejos — y terminamos en una neurona que decide *neumonía* o *normal*. El `historial` guarda cómo bajaba el error en cada época; si sube el error de validación mientras baja el de entrenamiento, es la señal de sobreajuste de la que avisa la unidad.


### B.4 Evaluamos: exactitud, AUC y matriz de confusión

Igual que con cualquier clasificador de este curso, no basta con "entrenó bien": evaluamos sobre el **test oficial** (imágenes que la CNN no vio) con las métricas que ya conoces de U3 — exactitud, AUC y matriz de confusión — para saber si de verdad distingue neumonía de normalidad, y qué tipo de errores comete.


In [ ]:
try:
    if not MEDMNIST_OK:
        raise RuntimeError("no hay CNN entrenada (ver celdas anteriores)")

    from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt

    prob_cnn = cnn.predict(Xte_img, verbose=0).ravel()
    pred_cnn = (prob_cnn >= 0.5).astype(int)

    exactitud = (pred_cnn == yte_img).mean()
    auc_cnn = roc_auc_score(yte_img, prob_cnn)
    print("Exactitud en test:", round(exactitud, 3))
    print("AUC en test      :", round(auc_cnn, 3))

    cm = confusion_matrix(yte_img, pred_cnn)
    ConfusionMatrixDisplay(cm, display_labels=["normal", "neumonía"]).plot(cmap="Blues")
    plt.title("Matriz de confusión — CNN sobre PneumoniaMNIST")
    plt.show()

except Exception as e:
    print("Esta celda está pensada para Google Colab con GPU; aquí se omite:",
          "no hay una CNN entrenada sobre la que evaluar (ver el motivo en las celdas anteriores). Motivo:", repr(e))
    print("En Colab, esta celda mostraría la exactitud, el AUC y la matriz de confusión de la CNN sobre el test oficial de PneumoniaMNIST.")

**Cómo se leería este resultado:** una CNN pequeña como esta, entrenada en un par de minutos sobre PneumoniaMNIST, suele alcanzar una exactitud y un AUC notablemente altos — el conjunto está pensado para que un modelo modesto aprenda rápido. La matriz de confusión es la que de verdad importa en clínica: reparte los errores entre **falsos negativos** (radiografías con neumonía que el modelo llama normales — el error más grave) y **falsos positivos** (normales marcadas como sospechosas). Ningún número resume eso mejor que mirar la matriz completa.


{% hint style="danger" %}
**⚠️ Recuerda la advertencia clínica de la unidad**

Un AUC alto en PneumoniaMNIST **no es una validación clínica**. Es un ejercicio didáctico sobre un conjunto ya curado. Antes de fiarte de un modelo así en la práctica real habría que preguntar: ¿con qué radiografías se entrenó?, ¿se validó en otro hospital?, ¿en qué parte de la imagen se fija (mapas de calor tipo Grad-CAM)?, ¿rinde igual en todos los subgrupos? **"Funciona en el paper" no es "funciona en mi hospital".**
{% endhint %}


## Parte (C) · Usar un modelo ya entrenado (el puente hacia U9)

La idea central de esta unidad, y la que de verdad se usa en 2026, es esta: **ya casi nadie entrena redes desde cero**. Entrenar una CNN grande exige millones de imágenes etiquetadas; en medicina, donde etiquetar es caro y lento, eso rara vez existe. Lo normal es **partir de una red que alguien ya entrenó y afinó**, y simplemente usarla.

Vamos a hacer justo eso: cargar, sin entrenar nada, un modelo ya afinado para detectar neumonía en radiografías de tórax, publicado en **Hugging Face** — el "repositorio" de modelos de IA ya entrenados que veremos a fondo en **U9**. Con la librería `transformers` y su función `pipeline`, usar un modelo de imagen se resuelve en unas pocas líneas.


### C.1 Cargamos el modelo ya entrenado

Pedimos un `pipeline` de tipo `"image-classification"` con el modelo `dima806/chest_xray_pneumonia_detection`, ya afinado por otra persona sobre radiografías de tórax. Esto requiere instalar `transformers` y descargar los pesos del modelo (unos cientos de MB) la primera vez — de ahí, otra vez, el `try/except`: en Colab con red hay descarga real; aquí, si falta red, se avisa y se sigue.


In [ ]:
try:
    try:
        import transformers, torch  # Colab ya trae torch de fábrica; si falta, lo instalamos
        from PIL import Image
    except ImportError:
        import subprocess, sys as _sys
        resultado = subprocess.run(
            [_sys.executable, "-m", "pip", "install", "-q", "transformers", "torch", "pillow"],
            capture_output=True, text=True, timeout=20,
        )
        if resultado.returncode != 0:
            raise RuntimeError(resultado.stderr[-300:] if resultado.stderr else "pip devolvió un error")
        import transformers, torch
        from PIL import Image

    from transformers import pipeline
    clasificador_rx = pipeline(
        "image-classification",
        model="dima806/chest_xray_pneumonia_detection",
    )
    HF_OK = True
    print("Modelo cargado correctamente: dima806/chest_xray_pneumonia_detection")
    print("Este modelo NO lo hemos entrenado nosotros: alguien ya hizo el trabajo pesado.")

except Exception as e:
    HF_OK = False
    print("Esta celda está pensada para Google Colab con GPU; aquí se omite:",
          "no se ha podido instalar transformers/torch o descargar el modelo (sin red, timeout, o dependencias no disponibles). Motivo:", repr(e))
    print("En Colab, ejecuta: !pip install transformers torch pillow  — y la descarga del modelo se completa en segundos.")

**Qué acabamos de hacer (en Colab):** con cuatro líneas tenemos un clasificador de radiografías **ya entrenado por otra persona**, sin escribir una sola capa de red ni entrenar nada. Esta es, literalmente, la forma en que la mayoría de proyectos de imagen médica arrancan hoy: **no se construye una CNN desde cero — se busca un modelo afín ya publicado y se evalúa si sirve** (o se afina un poco más con datos propios, el *fine-tuning* de la sección 8.4 de la unidad).


### C.2 Lo probamos sobre una imagen de PneumoniaMNIST

Si la parte (B) llegó a descargar PneumoniaMNIST, reutilizamos una de sus imágenes de test para pasarla por el modelo de Hugging Face y comparar su predicción con la etiqueta real. Si no hay imágenes disponibles (por ejemplo, porque tampoco hubo red en la parte B), lo explicamos igualmente sin romper el cuaderno.


In [ ]:
try:
    if not HF_OK:
        raise RuntimeError("el modelo de Hugging Face no está disponible (ver celda anterior)")
    if not MEDMNIST_OK:
        raise RuntimeError("no hay imágenes de PneumoniaMNIST cargadas (ver parte B)")

    from PIL import Image
    import numpy as np

    img_array, etiqueta_real = test_ds[0]
    img_pil = Image.fromarray(np.array(img_array)).convert("RGB").resize((224, 224))

    prediccion = clasificador_rx(img_pil)
    nombre_real = medmnist.INFO["pneumoniamnist"]["label"][str(int(etiqueta_real[0]))]

    print("Etiqueta real de la imagen :", nombre_real)
    print("Predicción del modelo ya entrenado:")
    for p in prediccion:
        print(f"  {p['label']}: {p['score']:.3f}")

except Exception as e:
    print("Esta celda está pensada para Google Colab con GPU; aquí se omite:",
          "no se ha podido ejecutar el modelo sobre una imagen (falta el modelo, faltan las imágenes, o no hay red). Motivo:", repr(e))
    print("En Colab, esta celda mostraría la etiqueta real de una radiografía de PneumoniaMNIST junto a la predicción del modelo ya entrenado, con su probabilidad.")

**Cómo se interpreta esto:** si la predicción coincide con la etiqueta real, es una señal de que el modelo preentrenado **generaliza razonablemente bien** a este conjunto, aunque nunca lo hayamos entrenado ni afinado nosotros con estos datos concretos. Ojo: este modelo se entrenó con **sus propias** radiografías, de **su propio** origen y equipo — el aviso de la unidad sobre atajos espurios y validación externa aplica aquí con toda su fuerza: que funcione bien en una imagen suelta no dice nada sobre si funcionaría bien en tu hospital, con tu aparato, con tu población.


{% hint style="success" %}
**💡 Idea clave de la parte (C) — el puente a U9**

Esto que hemos hecho en dos celdas —cargar un modelo ya entrenado por otros y usarlo con `pipeline(...)`— es exactamente el patrón que vertebra la **siguiente unidad**. Hugging Face es el repositorio donde vive una comunidad enorme de modelos ya afinados (de imagen, de texto, multimodales), y `pipeline` es la puerta de entrada más simple para usarlos. Cuanto más nos alejamos de la tabla y nos acercamos a imagen, señal o texto, más natural es que la pregunta no sea "¿cómo entreno esta red?" sino **"¿qué modelo ya entrenado por otros me sirve, y cómo lo valido con criterio clínico antes de fiarme?"**.
{% endhint %}


## Qué hemos aprendido

* Un **MLP** (red neuronal básica) entrenado sobre `pacientes.csv` **no mejora** a la regresión logística (AUC ≈ 0,84 ambos): en datos **tabulares**, sigue ganando lo simple. Las redes no son "mejores" en abstracto — son la herramienta adecuada **cuando el dato tiene forma de imagen o de señal**.
* Con imagen médica real (**PneumoniaMNIST**), una **CNN** pequeña sí explota algo que el MLP tabular no tenía: la **estructura espacial** de los píxeles, aprendiendo sola una jerarquía de patrones (bordes → texturas → estructuras).
* En 2026, la vía habitual **no es entrenar desde cero**: es partir de un **modelo ya entrenado** (transfer learning, o directamente un modelo afinado publicado en Hugging Face) y adaptarlo o evaluarlo con criterio clínico. Es el puente natural hacia **U9** (modelos fundacionales).
* Cualquier resultado brillante en un cuaderno como este es un punto de partida, no una validación: faltan la **validación externa**, la revisión de **atajos espurios** y el análisis **por subgrupos** de los que habla la unidad.

**Para seguir explorando con un asistente de IA**, podrías pedirle algo como:

> *"Con PneumoniaMNIST (MedMNIST), en español y por celdas, con PyTorch o Keras y GPU si está disponible: (1) entrena una CNN sencilla desde cero y reporta accuracy, AUC y matriz de confusión sobre el test oficial; (2) repite con transfer learning desde una red preentrenada (congelando las capas base) y compara resultados y tiempo de entrenamiento; (3) muéstrame un mapa de calor tipo Grad-CAM de un par de casos para ver en qué zona de la imagen se fija el modelo; (4) coméntame qué riesgos de 'atajo espurio' habría que vigilar y por qué esto no estaría listo para un hospital sin validación externa."*

La próxima unidad, **U9**, es precisamente sobre eso: de dónde salen estos modelos ya entrenados, cómo se buscan y se prueban en tres líneas, y cómo se manejan con criterio.
